# Notebook 5: CI/CD para Prompts — Del Commit al Deploy sin Tocar Código

## Objetivos de aprendizaje
- Entender por qué el **prompt necesita su propio pipeline** (separado del código)
- Construir el flujo CI/CD completo: **push → evaluate → promote**
- Configurar **GitHub Actions** y **GitLab CI** para automatizar el flujo
- Implementar la app **Streamlit con pestañas por entorno** (dev/staging/prod)
- Dominar el concepto de **labels como punteros a entornos** en Langfuse

## Concepto clave

> **El CI/CD es para el PROMPT, no para el código.**
> El código del agente vive en Git y se ejecuta localmente o en un servidor fijo.
> El prompt es el artefacto que cambia frecuentemente y necesita su propio pipeline.

```
CÓDIGO (Git)                         PROMPT (Langfuse)
─────────────                        ─────────────────
- Cambia poco                        - Cambia frecuentemente
- Deploy = rebuild + restart         - Deploy = mover un label (0 downtime)
- Rollback = git revert + redeploy   - Rollback = mover label atrás (5 seg)
- Test = pytest (determinista)       - Test = evaluación semántica
- CI/CD = GitHub Actions / GitLab    - CI/CD = scripts Python + Langfuse API
```

> **Referencia:**
> - [Langfuse Prompt Management](https://langfuse.com/docs/prompt-management/overview)
> - [Langfuse Prompt Version Control](https://langfuse.com/docs/prompt-management/features/prompt-version-control)

---

## Parte 1: Arquitectura del flujo CI/CD para Prompts

### El diagrama completo

```
  Desarrollador edita prompts/system_prompt.txt
          │
          ▼
    ┌──────────────────┐
    │   git push        │  ← Trigger del pipeline
    └────────┬─────────┘
             │
  ╔══════════╧════════════════════════════════════════════╗
  ║                    PIPELINE CI/CD                      ║
  ║                                                        ║
  ║  ┌─────────────────────────┐                           ║
  ║  │  1. PUSH (CI)           │                           ║
  ║  │  push_prompt.py         │                           ║
  ║  │  → Langfuse: staging    │                           ║
  ║  └───────────┬─────────────┘                           ║
  ║              │                                         ║
  ║  ┌───────────▼─────────────┐                           ║
  ║  │  2. EVALUATE (QG)       │                           ║
  ║  │  evaluate_prompt.py     │                           ║
  ║  │  → run_experiment()     │                           ║
  ║  │  → exit 0 / exit 1     │                           ║
  ║  └───────────┬─────────────┘                           ║
  ║              │                                         ║
  ║        ┌─────┴─────┐                                   ║
  ║     PASS ✅     FAIL ❌                                ║
  ║        │           │                                   ║
  ║  ┌─────▼─────┐  ┌──▼────────────┐                     ║
  ║  │ 3. PROMOTE│  │ Pipeline STOP │                     ║
  ║  │ promote   │  │ production    │                     ║
  ║  │ _prompt.py│  │ sin cambios   │                     ║
  ║  └───────────┘  └───────────────┘                     ║
  ╚════════════════════════════════════════════════════════╝
          │
          ▼
    ┌──────────────────────────────────┐
    │  STREAMLIT (CD dinámico)          │
    │  Cada pestaña = un label distinto │
    │  Dev → "latest"                   │
    │  Staging → "staging"              │
    │  Production → "production"        │
    │                                   │
    │  get_prompt(name, label=...)      │
    │  → devuelve la versión correcta   │
    └──────────────────────────────────┘
```

### ¿Por qué esta arquitectura funciona?

| Decisión | Razón |
|----------|-------|
| **Scripts Python** (no bash) | Descriptivos, testeables, portables entre GitHub/GitLab |
| **Labels como entornos** | Desacopla versión de entorno: misma versión puede estar en staging y prod |
| **Langfuse como registry** | El prompt se almacena fuera del código; cambio = mover un puntero |
| **Evaluación antes de promover** | Impide que un prompt defectuoso llegue a producción |
| **Streamlit con tabs** | Permite al equipo ver el prompt activo en cada entorno en tiempo real |

In [ ]:
# Setup
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

from langfuse import get_client
langfuse = get_client()
langfuse.auth_check()
print("✅ Conectado a Langfuse")

---

## Parte 2: El script CI — `push_prompt.py`

### ¿Qué hace?
Lee un archivo de texto con el prompt y lo sube a Langfuse como nueva versión inmutable con los labels especificados.

### Ubicación
`src/techshop_agent/cicd/push_prompt.py`

### Uso desde terminal
```bash
# Desde un archivo
python -m techshop_agent.cicd.push_prompt \
    --file prompts/system_prompt.txt \
    --labels staging latest

# Contenido inline (útil para tests)
python -m techshop_agent.cicd.push_prompt \
    --content "Eres Alex, asistente de TechShop..." \
    --labels staging
```

### ¿Qué ocurre por debajo?
1. Lee el contenido del archivo (o del argumento `--content`)
2. Conecta a Langfuse usando `LANGFUSE_PUBLIC_KEY` y `LANGFUSE_SECRET_KEY`
3. Llama a `langfuse.create_prompt(name, prompt, labels, type="text")`
4. Langfuse asigna automáticamente un número de versión (v1, v2, v3...)
5. El prompt queda **inmutable** — no se puede editar después de crearlo

> **Clave:** El script NO asigna label `production`. Solo asigna `staging` y `latest`. El label `production` se asigna DESPUÉS de que la evaluación pase.

In [ ]:
# Demo: ejecutar push_prompt directamente
from techshop_agent.cicd.push_prompt import push_prompt
from pathlib import Path

# Leer el prompt del archivo en el repo
prompt_file = Path("../../prompts/system_prompt.txt")
content = prompt_file.read_text(encoding="utf-8")

print(f"📄 Prompt file: {prompt_file}")
print(f"   Longitud: {len(content)} caracteres")
print(f"   Primeras líneas:")
for line in content.split("\n")[:5]:
    print(f"   │ {line}")
print(f"   │ ...")

# Subir a Langfuse con label staging
result = push_prompt(
    content=content,
    labels=["staging", "latest"],
    config={"author": "notebook-demo", "description": "Demo from NB05"},
)
print(f"\n✅ Subido: {result}")

---

## Parte 3: GitHub Actions — Pipeline completo

### El archivo `.github/workflows/prompt-cicd.yml`

Este workflow se ejecuta automáticamente cuando hay cambios en la carpeta `prompts/`.
Tiene 3 jobs encadenados:

```yaml
on:
  push:
    paths: ['prompts/**']  ← Solo se ejecuta si cambian prompts

jobs:
  push-to-langfuse:     ← CI: sube prompt
  evaluate:             ← Quality Gate: evalúa
  promote:              ← CD: promueve si pasa
    needs: [evaluate]   ← Solo corre si evaluate pasa
```

### ¿Cómo configurar los secrets?

En GitHub: Settings → Secrets and variables → Actions → New repository secret:
- `LANGFUSE_PUBLIC_KEY` — pk-lf-...
- `LANGFUSE_SECRET_KEY` — sk-lf-...
- `LANGFUSE_BASE_URL` — https://cloud.langfuse.com
- `AWS_ACCESS_KEY_ID` — para Bedrock
- `AWS_SECRET_ACCESS_KEY` — para Bedrock
- `AWS_REGION` — eu-west-1

In [ ]:
# Generar y mostrar el archivo de GitHub Actions
github_workflow = """\
# =============================================================================
# Prompt CI/CD Pipeline — GitHub Actions
# =============================================================================
# Este workflow automatiza el ciclo de vida del prompt del agente TechShop:
#   1. Push: sube el prompt a Langfuse como nueva versión (staging)
#   2. Evaluate: ejecuta la suite de evaluación contra staging
#   3. Promote: si pasa, mueve el label "production" a la nueva versión
#
# Trigger: solo se ejecuta cuando cambian archivos en prompts/
# Secrets necesarios: LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL,
#                     AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION
# =============================================================================

name: Prompt CI/CD

on:
  push:
    paths:
      - 'prompts/**'        # Solo trigger si cambian prompts
  workflow_dispatch:         # Permite ejecución manual desde GitHub UI

env:
  PYTHON_VERSION: '3.11'

jobs:
  # ─── Job 1: Push del prompt a Langfuse ────────────────────────────────
  push-to-langfuse:
    name: 📤 Push Prompt to Langfuse
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install dependencies
        run: pip install -e ".[llmops]"

      - name: Push prompt to Langfuse (staging)
        env:
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
          LANGFUSE_BASE_URL: ${{ secrets.LANGFUSE_BASE_URL }}
        run: |
          python -m techshop_agent.cicd.push_prompt \\
            --file prompts/system_prompt.txt \\
            --labels staging latest \\
            --author "github-actions"

  # ─── Job 2: Evaluación (Quality Gate) ─────────────────────────────────
  evaluate:
    name: 🧪 Evaluate Prompt
    needs: push-to-langfuse
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install dependencies
        run: pip install -e ".[llmops]"

      - name: Run evaluation suite
        env:
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
          LANGFUSE_BASE_URL: ${{ secrets.LANGFUSE_BASE_URL }}
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          AWS_REGION: ${{ secrets.AWS_REGION }}
        run: |
          python -m techshop_agent.cicd.evaluate_prompt \\
            --label staging \\
            --threshold 0.7

  # ─── Job 3: Promoción a Production ────────────────────────────────────
  promote:
    name: 🚀 Promote to Production
    needs: evaluate           # Solo corre si evaluate pasa (exit 0)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install dependencies
        run: pip install -e ".[llmops]"

      - name: Promote staging → production
        env:
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
          LANGFUSE_BASE_URL: ${{ secrets.LANGFUSE_BASE_URL }}
        run: |
          python -m techshop_agent.cicd.promote_prompt \\
            --from-label staging \\
            --to-label production
"""

print(github_workflow)

---

## Parte 4: GitLab CI — Pipeline equivalente

### Mapeo de conceptos GitHub → GitLab

| GitHub Actions | GitLab CI | Equivalencia |
|---------------|-----------|-------------|
| `jobs:` | `stages:` + jobs | Estructura del pipeline |
| `needs: [job]` | `needs: [job]` | Dependencias entre jobs |
| `secrets.MY_KEY` | `$MY_KEY` (CI/CD Variables) | Credenciales seguras |
| `on: push: paths:` | `rules: changes:` | Trigger condicional |
| `runs-on: ubuntu-latest` | `image: python:3.11` | Entorno de ejecución |
| `actions/checkout@v4` | Automático | Checkout del repo |
| `actions/setup-python@v5` | `image: python:3.11` | Python version |

### Configurar variables en GitLab
Settings → CI/CD → Variables → Add variable:
- Tick "Mask variable" para secrets
- Mismas variables que en GitHub

In [ ]:
# Generar y mostrar el archivo de GitLab CI
gitlab_ci = """\
# =============================================================================
# Prompt CI/CD Pipeline — GitLab CI
# =============================================================================
# Equivalente al pipeline de GitHub Actions.
# Trigger: solo se ejecuta cuando cambian archivos en prompts/
# Variables CI/CD: LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL,
#                  AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION
# =============================================================================

stages:
  - push          # CI: sube prompt a Langfuse
  - evaluate      # Quality Gate: evalúa el prompt
  - promote       # CD: promueve a production

# ─── Variables globales ─────────────────────────────────────────────────
variables:
  PIP_CACHE_DIR: "$CI_PROJECT_DIR/.pip-cache"

# ─── Cache de pip (reutiliza entre pipelines) ───────────────────────────
.pip-cache: &pip-cache
  cache:
    key: pip-${CI_COMMIT_REF_SLUG}
    paths:
      - .pip-cache/

# ─── Job 1: Push del prompt a Langfuse ──────────────────────────────────
push-prompt:
  stage: push
  image: python:3.11-slim
  <<: *pip-cache
  rules:
    - changes:
        - prompts/**          # Solo trigger si cambian prompts
    - when: manual            # También permite ejecución manual
  script:
    - pip install -e ".[llmops]"
    - python -m techshop_agent.cicd.push_prompt
        --file prompts/system_prompt.txt
        --labels staging latest
        --author "gitlab-ci"

# ─── Job 2: Evaluación (Quality Gate) ──────────────────────────────────
evaluate-prompt:
  stage: evaluate
  image: python:3.11-slim
  <<: *pip-cache
  needs:
    - push-prompt
  script:
    - pip install -e ".[llmops]"
    - python -m techshop_agent.cicd.evaluate_prompt
        --label staging
        --threshold 0.7
  # Si evaluate_prompt.py retorna exit 1, GitLab marca el job como failed
  # y el pipeline se detiene — promote-prompt no se ejecuta.

# ─── Job 3: Promoción a Production ────────────────────────────────────
promote-prompt:
  stage: promote
  image: python:3.11-slim
  <<: *pip-cache
  needs:
    - evaluate-prompt        # Solo corre si evaluate-prompt pasa
  script:
    - pip install -e ".[llmops]"
    - python -m techshop_agent.cicd.promote_prompt
        --from-label staging
        --to-label production
  # Solo en la rama principal (no en feature branches)
  rules:
    - if: $CI_COMMIT_BRANCH == $CI_DEFAULT_BRANCH
"""

print(gitlab_ci)

---

## Parte 5: Streamlit con pestañas por entorno

### El concepto

La app Streamlit tiene **3 pestañas** que representan entornos diferentes. **Cada pestaña solo cambia el label** con el que se hace `get_prompt()`:

```python
# Pestaña "Production"  → langfuse.get_prompt(name, label="production")
# Pestaña "Staging"     → langfuse.get_prompt(name, label="staging")
# Pestaña "Development" → langfuse.get_prompt(name, label="latest")
```

Esto es el **CD dinámico**: la app no necesita re-deployarse cuando cambia el prompt. Langfuse devuelve la versión que tenga el label correspondiente.

### Implementación clave

```python
ENVIRONMENTS = {
    "🟢 Production": "production",
    "🟡 Staging":    "staging",
    "🔵 Development": "latest",
}

tabs = st.tabs(list(ENVIRONMENTS.keys()))
for tab, (env_name, label) in zip(tabs, ENVIRONMENTS.items()):
    with tab:
        # Fetch prompt con este label
        prompt = langfuse.get_prompt("techshop-system-prompt", label=label)
        st.code(prompt.compile(), language="text")
        
        # El agente usa este prompt
        agent = create_agent(system_prompt=prompt.compile())
```

> **Nota:** En la pestaña "Production" el usuario ve lo que ven los clientes reales. En "Staging" se prueba el candidato. En "Development" se ve la última versión creada.

---

## Parte 6: Resumen — Todo el contenido del Día 2

### Archivos creados hoy

| Archivo | Propósito |
|---------|----------|
| `src/techshop_agent/evaluation.py` | Dataset + evaluadores + runner + CLI |
| `src/techshop_agent/cicd/push_prompt.py` | CI: sube prompt a Langfuse |
| `src/techshop_agent/cicd/evaluate_prompt.py` | Quality Gate: evalúa y devuelve exit code |
| `src/techshop_agent/cicd/promote_prompt.py` | CD: mueve label a production |
| `prompts/system_prompt.txt` | El prompt como artefacto versionable |
| `.github/workflows/prompt-cicd.yml` | GitHub Actions pipeline |
| `.gitlab-ci.yml` | GitLab CI pipeline |
| `streamlit_app/app.py` | App con pestañas por entorno |

### Comandos de terminal para recordar

```bash
# Push prompt
python -m techshop_agent.cicd.push_prompt --file prompts/system_prompt.txt --labels staging

# Evaluar
python -m techshop_agent.cicd.evaluate_prompt --label staging --threshold 0.7

# Promover
python -m techshop_agent.cicd.promote_prompt --from-label staging --to-label production

# Evaluación directa
python -m techshop_agent.evaluation --label production

# Streamlit
cd streamlit_app && streamlit run app.py
```

### El ciclo LLMOps completo

```
Día 1: Construir → Observar → Versionar
Día 2: Evaluar → Automatizar → CI/CD
         ✅ COMPLETADO
```

> **Referencia oficial:**
> - [Langfuse Prompt Management](https://langfuse.com/docs/prompt-management/overview)
> - [Langfuse Evaluation](https://langfuse.com/docs/evaluation/overview)
> - [Langfuse Experiments via SDK](https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk)